In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np

In [2]:
mat_list = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

df = pd.read_csv("../../data/FRB_H15.csv").dropna()
df.columns = ["Date", *mat_list]
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

In [3]:
# Estimate the three Nelson-Siegel beta factors for each yield curve.
maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
lambda_ns = 0.0609
loadings = np.array([
    [
        1,
        (1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity),
        ((1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity))
        - np.exp(-lambda_ns * maturity),
    ]
    for maturity in maturities
])

beta_values = [
    np.linalg.lstsq(loadings, yields, rcond=None)[0]
    for yields in df.to_numpy()
]
betas = pd.DataFrame(
    beta_values,
    index=df.index,
    columns=["Beta 1", "Beta 2", "Beta 3"],
)

In [4]:
# Evaluate exactly 20 expanding, one-day-ahead rolling forecasts.
rolling_windows = 20
forecast_indexes = range(len(betas) - rolling_windows, len(betas))

def rolling_metrics(errors, predicted_changes, actual_changes):
    rmse_basis_points = 100 * np.sqrt(np.mean(np.asarray(errors) ** 2))
    directional_accuracy = 100 * np.mean(
        np.sign(predicted_changes) == np.sign(actual_changes)
    )
    return rmse_basis_points, directional_accuracy

beta_metrics = []

for beta_index in range(3):
    errors = []
    predicted_changes = []
    actual_changes = []

    for forecast_index in forecast_indexes:
        history = betas.iloc[:forecast_index, beta_index].to_numpy()
        x_train = history[:-1].reshape(-1, 1)
        y_train = history[1:]
        model = xgb.XGBRegressor(
            objective="reg:squarederror",
            random_state=7,
        )
        model.fit(x_train, y_train)

        current = history[-1]
        predicted = model.predict(np.array([[current]]))[0]
        actual = betas.iloc[forecast_index, beta_index]

        errors.append(predicted - actual)
        predicted_changes.append(predicted - current)
        actual_changes.append(actual - current)

    beta_metrics.append(
        rolling_metrics(errors, predicted_changes, actual_changes)
    )

In [5]:
print(f"Rolling windows: {rolling_windows}")
for beta, (beta_rmse, directional_accuracy) in enumerate(beta_metrics, start=1):
    print(f"  Beta: {beta}")
    print(
        f"  Forecast 1 day, RMSE: {beta_rmse} basis points, "
        f"Directional Accuracy: {directional_accuracy}%"
    )

Rolling windows: 20
  Beta: 1
  Forecast 1 day, RMSE: 28.405200910421456 basis points, Directional Accuracy: 70.0%
  Beta: 2
  Forecast 1 day, RMSE: 30.852599416104347 basis points, Directional Accuracy: 25.0%
  Beta: 3
  Forecast 1 day, RMSE: 54.69055699636443 basis points, Directional Accuracy: 40.0%
